# Inference Aksara Lontara
## Load model hasil training & prediksi gambar dari Google Drive

**Cara pakai:**
1. Upload gambar aksara lontara ke folder `datatest/` di Google Drive (`MyDrive/ImageCaptioning/datatest/`)
2. Pastikan folder `model/blip_lontara/best_model/` sudah ada di Google Drive
3. Jalankan notebook ini

In [ ]:
import os
import torch
import math
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import matplotlib.pyplot as plt

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Konfigurasi
DRIVE_DIR  = '/content/drive/MyDrive/ImageCaptioning'
MODEL_DIR  = f'{DRIVE_DIR}/model/blip_lontara/best_model'
IMAGE_DIR  = f'{DRIVE_DIR}/datatest'
MAX_LENGTH = 64
NUM_BEAMS  = 4
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

# Load model
assert os.path.exists(MODEL_DIR), f'Model tidak ditemukan di {MODEL_DIR}!'
print(f'Memuat model dari: {MODEL_DIR}')
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()
print('\u2705 Model siap!')


def prediksi(image_path):
    """Prediksi caption dari satu gambar."""
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_length=MAX_LENGTH, num_beams=NUM_BEAMS,
            early_stopping=True, no_repeat_ngram_size=2
        )
    return processor.decode(output[0], skip_special_tokens=True)


# Prediksi semua gambar di folder datatest
VALID_EXT = ('.png', '.jpg', '.jpeg')
assert os.path.exists(IMAGE_DIR), f'Folder "{IMAGE_DIR}" tidak ditemukan! Buat dulu di Google Drive.'
images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(VALID_EXT)])
assert len(images) > 0, f'Folder "{IMAGE_DIR}" kosong! Taruh gambar aksara lontara di sana.'
print(f'\nDitemukan {len(images)} gambar di folder datatest\n')

results = []
for img_name in images:
    img_path = os.path.join(IMAGE_DIR, img_name)
    caption = prediksi(img_path)
    results.append({'gambar': img_name, 'caption': caption})
    print(f'  {img_name} \u2192 {caption}')

# Visualisasi (grid otomatis sesuai jumlah gambar, maks 12)
show = results[:12]
cols = min(3, len(show))
rows = math.ceil(len(show) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if rows * cols == 1:
    axes = [axes]
else:
    axes = axes.flatten()
for idx, item in enumerate(show):
    img = Image.open(os.path.join(IMAGE_DIR, item['gambar']))
    axes[idx].imshow(img)
    axes[idx].set_title(item['caption'], fontsize=9)
    axes[idx].axis('off')
for idx in range(len(show), len(axes)):
    axes[idx].axis('off')
plt.tight_layout()
plt.show()

print(f'\n\u2705 Selesai! Total {len(results)} gambar diproses.')